#Developing a RAG application using Langchain   

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [53]:
import os
from dotenv import load_dotenv
load_dotenv("/Users/admin/Documents/rag/RAG_Using_Langchain/.env")

from google import genai
from google.genai import types

from IPython.display import display
from IPython.display import Markdown
import textwrap

from pathlib import Path
from pypdf import PdfReader

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import  GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_chroma import Chroma
from langchain_classic.chains import RetrievalQA

import psycopg

In [3]:
model = "gemini-3.6-flash"
client = genai.Client()

In [26]:
response = client.models.generate_content(
    model=model,
    contents="Explain Generative AI with 3 bullet points")
print(response.text)
Markdown(response.text)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Here is Generative AI explained in 3 bullet points:

*   **Creates Brand-New Content:** Unlike traditional AI that only analyzes or categorizes existing data, Generative AI produces entirely new, human-like content—including text, images, music, video, and computer code.
*   **Learns from Massive Patterns:** It works by using advanced deep learning models trained on vast datasets, allowing the AI to understand underlying patterns and predict what should come next when given a prompt.
*   **Powers Modern Everyday Tools:** It is the core technology behind popular applications like ChatGPT (for writing and brainstorming), Midjourney or DALL-E (for art and design), and GitHub Copilot (for programming).


Here is Generative AI explained in 3 bullet points:

*   **Creates Brand-New Content:** Unlike traditional AI that only analyzes or categorizes existing data, Generative AI produces entirely new, human-like content—including text, images, music, video, and computer code.
*   **Learns from Massive Patterns:** It works by using advanced deep learning models trained on vast datasets, allowing the AI to understand underlying patterns and predict what should come next when given a prompt.
*   **Powers Modern Everyday Tools:** It is the core technology behind popular applications like ChatGPT (for writing and brainstorming), Midjourney or DALL-E (for art and design), and GitHub Copilot (for programming).

In [ ]:
hist = model.start_chat()
response = hist.send_message("Hi! Give me a recipe to make a margeritta pizza from scratch.")
Markdown(response.text)

for i in hist.history:
    print(i)
    print('\n\n')
i.parts[0].text 

model.count_tokens("Now please help me find the nearest supermarket from where I can buy the ingredients.")

In [ ]:
def get_response(prompt, generation_config={}):
    response = model.generate_content(contents=prompt, 
    generation_config=generation_config)
    return response

for temp in [0.0, 0.25, 0.5, 0.75, 1.0]:
  config = genai.types.GenerationConfig(temperature=temp)
  result = get_response("Explain the concepts of XGBoost and Random Forest with real-life use cases", generation_config=config)

  print(f"\n\nFor temperature value {temp}, the results are: \n\n")
  display(Markdown(result.text))

In [ ]:
def get_response(prompt, generation_config={}):
    response = model.generate_content(contents=prompt, generation_config=generation_config)
    return response

for m_o_tok in [1000, 2000, 7000]:
    config = genai.types.GenerationConfig(max_output_tokens=m_o_tok)
    
    print(f"\n\n--- Max Output Tokens: {m_o_tok} ---")
    try:
        # Get the result
        result = get_response("Explain the concepts of XGBoost and Random Forest with real-life use cases", generation_config=config)
        
        # Only print text if the model actually returned text parts
        if result.candidates[0].content.parts:
            display(Markdown(result.text))
        # Handle the empty response
        # This is because different models have different MOTok to even start generating the answer
        else: 
            print(f"Result was empty! The model stopped because: {result.candidates[0].finish_reason}")
            print("(Reason 2 means 'MAX_TOKENS' was reached before a complete word could be formed)")
            
    except Exception as e:
        print(f"Skipped due to error: {e}")

In [ ]:
def get_response(prompt, generation_config={}):
    response = model.generate_content(contents=prompt, 
    generation_config=generation_config)
    return response

for k in [1, 4, 16, 32, 40]:
    config = genai.types.GenerationConfig(top_k=k)
    result = get_response("Explain the concepts of XGBoost and Random Forest with real-life use cases", generation_config=config)

    print(f"\n\nFor top k value {k}, the results are: \n\n")
    display(Markdown(result.text))

# Demo for Retrieval Augmented Generation

Step 1: Load a PDF and extract its text

In [19]:
from langchain_community.document_loaders import PyPDFLoader

# Initialize the loader with the path to your PDF file
pdfloader = PyPDFLoader("./whatisai.pdf")

# Load the PDF and split it into pages
pdf_pages = pdfloader.load()

print(f"Total pages loaded: {len(pdf_pages)}")

# Declare chunk size and overlap for text splitting
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200


# Splitting text into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
context = "\n\n".join(str(p.page_content) for p in pdf_pages)
texts = text_splitter.split_text(context)
print(f"Total chunks created: {len(texts)}")
print(f"First chunk: {texts[0]}")
print(f"Second chunk: {texts[1]}")
print(f"Third chunk: {texts[2]}")

Total pages loaded: 15
Total chunks created: 39
First chunk: Extinguished philosophies lie about the cradle of every sci ence as the
strangled snakes beside that of Hercules. - adapted from T. H . Huxley
1
Second chunk: WHAT IS ARTIFICIAL INTELLIGENCE?
John McCarthy
Computer Science Department
Stanford University
Stanford, CA 94305
jmc@cs.stanford.edu
http://www-formal.stanford.edu/jmc/
2007 Nov 12, 2:05 a.m.
Revised November 12, 2007:
Abstract
This article for the layman answers basic questions about ar tiﬁcial
intelligence. The opinions expressed here are not all conse nsus opinion
among researchers in AI.
1 Basic Questions
Q. What is artiﬁcial intelligence?
A. It is the science and engineering of making intelligent ma chines, es-
pecially intelligent computer programs. It is related to th e similar task of
using computers to understand human intelligence, but AI do es not have to
conﬁne itself to methods that are biologically observable.
Q. Yes, but what is intelligence?
A. Intellig

Step 2: Create the Gemini model and generate embeddings

In [20]:
# Initiate the model and embeddings with the API key from environment variables
API_KEY = os.getenv("GEMINI_API_KEY")

# Create the model and embeddings instances
gemini_model = ChatGoogleGenerativeAI(model=model, google_api_key=API_KEY, temperature=0.8)

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001", api_key=API_KEY)


Use Chroma for vector Database

In [21]:
# Create a vector database and a retriever for querying
vector_index = Chroma.from_texts(texts, embeddings)
retriever = vector_index.as_retriever(search_kwargs={"k" : 5})

In [22]:
# Create a RAG question-answering chain using the model and retriever
qa_chain = RetrievalQA.from_chain_type(gemini_model, retriever=retriever, return_source_documents=True)

In [23]:
# Example usage 
user_query = "Name few good reading materials for learning AI."
result = qa_chain.invoke({"query": user_query})
print("Answer:", result["result"])

/Users/admin/Documents/rag/RAG_Using_Langchain/.venv/lib/python3.11/site-packages/langchain_google_genai/chat_models.py:3237: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Answer: Based on the provided text, here are several recommended reading materials/textbooks for learning AI:

* ***Artificial Intelligence* by Stuart Russell and Peter Norvig** (Prentice Hall) – Noted as the most commonly used textbook in 1997.
* ***Artificial Intelligence: A New Synthesis* by Nils Nilsson** (Morgan Kaufmann) – Mentioned as a potentially easier option to read.
* ***Computational Intelligence* by David Poole, Alan Mackworth, and Randy Goebel** (Oxford, 1998).
* ***Machine Learning* by Tom Mitchell** (McGraw-Hill, 1997).


Create embeddings and store them in ChromaDB (local storage)

In [ ]:
# LangChain automatically gets or creates the collection for you

import uuid


vector_store = Chroma(
    collection_name="my_documents",
    embedding_function=embeddings,
    persist_directory="./chroma_db",
    collection_metadata={"hnsw:space": "cosine"} # This sets up cosine similarity
)

# Add documents using the Chroma API supported by this installed LangChain version
print("--- STEP 0: Adding Documents to the Collection ---")
documents = [
    Document(page_content=doc.page_content, metadata={"source": "whatisai.pdf"})
    for doc in pdf_pages
]
unique_ids = [str(uuid.uuid4()) for _ in documents]

print("Uploading documents to ChromaDB...")
vector_store.add_documents(documents=documents, ids=unique_ids)

stored_docs = vector_store.get()
print(f"Successfully loaded data! Total items now in collection: {len(stored_docs['ids'])}")



--- STEP 0: Adding Documents to the Collection ---
Uploading documents to ChromaDB...
Successfully loaded data! Total items now in collection: 15


In [ ]:
user_query = "Name few good reading materials for learning AI."

SIMILARITY_THRESHOLD = 0.70
# --- METHOD A: Direct Search with Scores ---
print("\n--- LangChain Score Search ---")
results = vector_store.similarity_search_with_relevance_scores(
    query=user_query,
    k=3,
    score_threshold=SIMILARITY_THRESHOLD  # LangChain handles the distance conversion for you!
)

for doc, score in results:
    print(f"Similarity Score: {score:.4f} | Content: {doc.page_content[:200]}...")



--- LangChain Score Search ---
Similarity Score: 0.7560 | Content: knowing the languages currently in fashion. In the late 1990 s, these include
C++ and Java.
Q. What is a good textbook on AI?
A. Artiﬁcial Intelligence by Stuart Russell and Peter Norvig, Prentice Hal...
Similarity Score: 0.7182 | Content: 9.13
References
[Den98] Daniel Dennett. Brainchildren: Essays on Designing Minds . MIT
Press, 1998.
[Jen98] Arthur R. Jensen. Does IQ matter? Commentary, pages 20–21,
November 1998. The reference is j...
Similarity Score: 0.7116 | Content: been previous credit card frauds at this establishment).
4 More questions
Q. How is AI research done?
A. AI research has both theoretical and experimental sides. The experi-
mental side has both basic...


In [44]:
# --- METHOD B: Build as an Automated RAG Retriever, you dont need the score ---
retriever = vector_store.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"score_threshold": SIMILARITY_THRESHOLD, "k": 3}
)

retrieved_contexts = retriever.invoke(user_query)
print("\n--- LangChain RAG Retriever ---")
for doc in retrieved_contexts:
    print(f"Content: {doc.page_content[:200]}...")
    


--- LangChain RAG Retriever ---
Content: knowing the languages currently in fashion. In the late 1990 s, these include
C++ and Java.
Q. What is a good textbook on AI?
A. Artiﬁcial Intelligence by Stuart Russell and Peter Norvig, Prentice Hal...
Content: 9.13
References
[Den98] Daniel Dennett. Brainchildren: Essays on Designing Minds . MIT
Press, 1998.
[Jen98] Arthur R. Jensen. Does IQ matter? Commentary, pages 20–21,
November 1998. The reference is j...
Content: been previous credit card frauds at this establishment).
4 More questions
Q. How is AI research done?
A. AI research has both theoretical and experimental sides. The experi-
mental side has both basic...


Use pgVector as vector database

# Use the following scripts to create the table that stores embeddings and create index:

CREATE TABLE documents (
    id SERIAL PRIMARY KEY,
    content TEXT NOT NULL,
    embedding VECTOR(3072),  -- 3072 dimensions for default gemini-embedding-001
    created_at TIMESTAMP DEFAULT NOW()
);

CREATE INDEX ON documents USING ivfflat (embedding vector_cosine_ops)

We use ivfflat index because Postgresql database cannot support more than 2000 dimensions for HNSW index

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv("/Users/admin/Documents/rag/RAG_Using_Langchain/.env")

import psycopg
from pgvector.psycopg import register_vector

DATABASE_URL = os.getenv("DATABASE_URL")
COLLECTION_NAME = "documents" # Ensure this matches the table name in your PostgreSQL database

def load_document_to_pgvector(pdf_pages):
    with psycopg.connect(DATABASE_URL) as conn:
        register_vector(conn)
        with conn.cursor() as cur:
            # Insert documents and their embeddings into the table
            for doc in pdf_pages:
                embedding_vector = embeddings.embed_query(doc.page_content)
                try:
                    cur.execute(f"""
                        INSERT INTO {COLLECTION_NAME} (content, embedding)
                        VALUES (%s, %s)
                        """, (doc.page_content, embedding_vector))
                except Exception as exc:
                    print(f"Failed to insert document: {exc}")
            conn.commit()


documents


In [51]:
load_document_to_pgvector(pdf_pages)   

In [58]:
from pgvector.psycopg import register_vector

DATABASE_URL = os.getenv("DATABASE_URL")
COLLECTION_NAME = "documents" # Ensure this matches the table name in your PostgreSQL database

def retrieve_relevant_context(user_query, top_k=3):
    context_chunks = []
    with psycopg.connect(DATABASE_URL) as conn:
        register_vector(conn)
        with conn.cursor() as cur:
            # Embed the user query
            query_embedding = embeddings.embed_query(user_query)
            
            # Retrieve the top_k most relevant documents based on cosine similarity
            cur.execute(f"""
                SELECT content, (1 - (embedding <=> %s::vector)) AS similarity
                FROM documents
                ORDER BY embedding <=> %s::vector ASC
                LIMIT %s;
                """, (query_embedding, query_embedding, top_k))
            
            results = cur.fetchall()

            for idx, (content, similarity) in enumerate(results, 1):
                # Print the score for debugging/monitoring
                print(f"Chunk #{idx} | Similarity Score: {similarity:.4f}")
                print(f"Snippet: {content[:100]}...\n")
                
                # Optional: Filter out weak matches (e.g., lower than 0.60)
                if similarity >= SIMILARITY_THRESHOLD:
                    context_chunks.append(content)
                else:
                    print(f"--> Chunk #{idx} dropped (below similarity threshold)")
    return context_chunks



In [66]:
user_query = "How to cook Pizza."

retrieved_contexts = retrieve_relevant_context(user_query, top_k=3)

print("Retrieved Contexts:")
print(retrieved_contexts)
    

Chunk #1 | Similarity Score: 0.5070
Snippet: been previous credit card frauds at this establishment).
4 More questions
Q. How is AI research done...

--> Chunk #1 dropped (below similarity threshold)
Chunk #2 | Similarity Score: 0.4866
Snippet: representation Facts about the world have to be represented in some way.
Usually languages of mathem...

--> Chunk #2 dropped (below similarity threshold)
Chunk #3 | Similarity Score: 0.4834
Snippet: knowing the languages currently in fashion. In the late 1990 s, these include
C++ and Java.
Q. What ...

--> Chunk #3 dropped (below similarity threshold)
Retrieved Contexts:
[]


Send the retrieved chunks to LLM to generate a response

In [75]:
# Send the retrieved contexts to the model for generating a response
SYSTEM_PROMPT = """You are a strict factual retrieval assistant operating under a Zero-Knowledge constraint. Your sole task is to answer user queries using ONLY the provided verified context.

### CRITICAL RULES:
1. NO PRE-TRAINED KNOWLEDGE: Do not use your own background knowledge, training data, assumptions, or external facts under any circumstances.
2. ABSOLUTE STRICTNESS: If a fact is not explicitly and literally stated in the provided context, treat it as completely untrue and non-existent.
3. TRUTHFULNESS: Never extrapolate, assume, guess, fill in gaps, or bridge concepts between sentences. 
4. NO FILLER OR PREAMBLES: Do not start answers with phrases like "Based on the context provided," "According to the documents," or "As requested." Just answer the question directly.

### LACK OF CONTEXT INTERFACES:
- If the context block is empty, completely blank, or missing, reply EXACTLY with: "I don't have enough information to answer your query."
- If the context block exists but does not contain the specific fact needed to answer the question, reply EXACTLY with: "I don't have enough information to answer your query."

### TEMPLATE FOR OUTPUT:
User Query: {user_query}
Answer: [Insert direct, factual answer synthesized solely from context here]
"""   
print("combined_context:", retrieved_contexts)
#if retrieved_contexts:
combined_context = "\n\n".join(retrieved_contexts)
prompt = f"User Query: {user_query}\n\nRelevant Context:\n{combined_context}\n\nPlease provide a comprehensive answer based on the above context."
    
response = client.models.generate_content(model = model, contents=prompt)
print("Model Response:")
print(response.text)

combined_context: []
Model Response:
Based on the provided context, there is no information available on how to cook a pizza.
